
# Domestic Regional Pilot — round22 실제 데이터 검증

`docs/DOMESTIC_REGIONAL_PILOT.md`(기술 상세명세서 7.5절을 뽑아 정리한 문서)가
서술하는 방법론과 수치를, 실제로 복구된
`data/v6_r22_snapshot/domestic_regional_pilot_v6.json`(100개 팬덤, round22
최종 상태)으로 재검증합니다. 이 파일럿은 이전에 정리한 Worldwide Language
Pilot과 달리 **원본 JSON과 원본 서술이 모두 이번 세션에 존재**하므로,
"재구성"이 아니라 **직접 재현·재검증**입니다.

1절은 RegionDiversity 공식과 데이터 무결성을, 2절은 문서가 인용한 v7
7라운드 시점 상위 18개 팬덤 표를 round22 최종 데이터로 다시 산출해
비교하고, 3절은 문서가 서술한 최종 상태(팬덤 커버리지 100/100, 검출 지역
수 총량 388)를 직접 확인합니다.


In [1]:

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")

with open(DATA_DIR / "domestic_regional_pilot_v6.json", encoding="utf-8") as f:
    drp = json.load(f)

REGIONS = ["서울","부산","대구","인천","광주","대전","울산","세종","경기","강원",
           "충북","충남","전북","전남","경북","경남","제주"]

print(f"domestic_regional_pilot_v6.json: {len(drp)}개 팬덤")
print("샘플(이영지):", json.dumps(drp["이영지"], ensure_ascii=False, indent=2))


domestic_regional_pilot_v6.json: 100개 팬덤
샘플(이영지): {
  "total_group_bullets": 79,
  "region_mention_counts": {
    "서울": 6,
    "부산": 3,
    "대구": 1,
    "인천": 1,
    "광주": 1,
    "대전": 0,
    "울산": 1,
    "세종": 0,
    "경기": 0,
    "강원": 0,
    "충북": 0,
    "충남": 2,
    "전북": 1,
    "전남": 0,
    "경북": 1,
    "경남": 1,
    "제주": 0
  },
  "total_region_mentions": 18,
  "n_regions_hit": 10,
  "region_diversity": 0.588,
  "primary_region": "서울",
  "primary_region_share": 0.333,
  "note": "TEXT-MINING PILOT: 이미 수집된 근거문장 내 국내 지역명(17개 시/도 및 대표 도시) 언급 횟수 기반. Coverage Index의 market_coverage(해외 시장)와는 별개 지표이며, 지역 연고(고향)와 지역 활동(투어·행사)을 구분하지 않고 합산한 1차 신호다."
}



## 1. 방법론 공식 재검증

문서 2절의 공식을 그대로 재계산합니다.

```
RegionDiversity(fandom) = n_regions_hit(검출된 서로 다른 지역 수) ÷ 17
```

또한 17개 지역별 언급 수의 합이 `total_region_mentions`와 일치하는지도
(기존 CSV 빌더 스크립트와 동일한 방식으로) 재확인합니다.


In [2]:

sum_mismatch = 0
diversity_mismatch = 0
n_regions_mismatch = 0

for fandom, d in drp.items():
    counts = d["region_mention_counts"]
    s = sum(counts.get(r, 0) for r in REGIONS)
    if s != d["total_region_mentions"]:
        sum_mismatch += 1

    recomputed_n_hit = sum(1 for r in REGIONS if counts.get(r, 0) > 0)
    if recomputed_n_hit != d["n_regions_hit"]:
        n_regions_mismatch += 1

    recomputed_div = round(d["n_regions_hit"] / 17, 3)
    if abs(recomputed_div - d["region_diversity"]) > 0.001:
        diversity_mismatch += 1

print(f"17개 지역 언급 합 != total_region_mentions 인 팬덤 수: {sum_mismatch} / {len(drp)}")
print(f"n_regions_hit 재계산 불일치 팬덤 수: {n_regions_mismatch} / {len(drp)}")
print(f"RegionDiversity(=n_regions_hit/17) 재계산 불일치(오차 >= 0.001) 팬덤 수: {diversity_mismatch} / {len(drp)}")


17개 지역 언급 합 != total_region_mentions 인 팬덤 수: 0 / 100
n_regions_hit 재계산 불일치 팬덤 수: 0 / 100
RegionDiversity(=n_regions_hit/17) 재계산 불일치(오차 >= 0.001) 팬덤 수: 0 / 100



## 2. 상위 팬덤 표 — round22 최종 데이터로 재산출 (v7 7라운드 표와 비교)

문서 3절의 표는 v7 7라운드(이 파일럿 신설 직후) 시점 값입니다. 이후 v7
8~22라운드에 걸쳐 지역 리서치가 여러 차례 더 누적됐으므로, round22 최종
데이터로 같은 표를 다시 만들어 무엇이 달라졌는지 봅니다.


In [3]:

rows = []
for fandom, d in drp.items():
    rows.append({
        "팬덤": fandom,
        "총 지역언급": d["total_region_mentions"],
        "검출 지역 수": d["n_regions_hit"],
        "요인 다양성": d["region_diversity"],
        "대표 지역": d["primary_region"],
        "대표 지역 비중": d["primary_region_share"],
        "근거 문장 수": d["total_group_bullets"],
    })

top_df = pd.DataFrame(rows).sort_values("총 지역언급", ascending=False).reset_index(drop=True)
top_df.index = top_df.index + 1
top_df.head(18)


,팬덤,총 지역언급,검출 지역 수,요인 다양성,대표 지역,대표 지역 비중,근거 문장 수
1,이승철,31,9,0.529,서울,0.129,31
2,싸이,30,9,0.529,강원,0.233,65
3,임영웅,27,12,0.706,서울,0.222,80
4,악동뮤지션,20,9,0.529,서울,0.300,45
5,김연자,20,8,0.471,광주,0.350,47
6,나훈아,20,8,0.471,서울,0.350,50
7,이영지,18,10,0.588,서울,0.333,79
8,다이나믹듀오,18,6,0.353,서울,0.278,42
9,조용필,18,6,0.353,서울,0.278,47
10,송가인,17,5,0.294,전남,0.706,51



문서 3절 표(v7 7라운드 시점)의 상위 3개는 이승철(31)·싸이(30)·임영웅(27)
이었습니다. round22 최종 표와 순위·수치를 직접 대조해, 이후 라운드의
누적 리서치가 실제로 반영됐는지 확인합니다.


In [4]:

doc_v7r7_top3 = {"이승철": 31, "싸이": 30, "임영웅": 27}
current_top3 = dict(zip(top_df["팬덤"].head(3), top_df["총 지역언급"].head(3)))

print("문서(v7 7라운드) 상위 3:", doc_v7r7_top3)
print("round22 최종 상위 3:", current_top3)
for name, v7r7_val in doc_v7r7_top3.items():
    current_val = drp[name]["total_region_mentions"]
    print(f"  {name}: v7 7라운드={v7r7_val} -> round22 최종={current_val} (증가 {current_val - v7r7_val}건)")


문서(v7 7라운드) 상위 3: {'이승철': 31, '싸이': 30, '임영웅': 27}
round22 최종 상위 3: {'이승철': 31, '싸이': 30, '임영웅': 27}
  이승철: v7 7라운드=31 -> round22 최종=31 (증가 0건)
  싸이: v7 7라운드=30 -> round22 최종=30 (증가 0건)
  임영웅: v7 7라운드=27 -> round22 최종=27 (증가 0건)



## 3. round22 최종 상태 — 문서 4절이 서술하는 두 최종 수치 확인

문서 4절은 두 가지를 최종 상태로 서술합니다: **① 팬덤 커버리지 100/100**
(v7 21라운드에서 도달), **② 검출 지역 수 총량 388**(목표 688의 약 56.4%,
v7 22라운드에서 포기·종료). 실제 데이터로 직접 확인합니다.


In [5]:

n_zero_hit = sum(1 for d in drp.values() if d["n_regions_hit"] == 0)
fandom_coverage = len(drp) - n_zero_hit
total_regions_hit_sum = sum(d["n_regions_hit"] for d in drp.values())

print(f"① 팬덤 커버리지: {fandom_coverage}/{len(drp)} (지역 언급 0건인 팬덤: {n_zero_hit}개)")
print(f"② 검출 지역 수 총량(n_regions_hit 합계): {total_regions_hit_sum} / 688 (목표 대비 {total_regions_hit_sum/688:.1%})")

print()
print("문서 서술과 일치 여부:")
print(f"  팬덤 커버리지 100/100 일치 = {fandom_coverage == 100}")
print(f"  검출 지역 수 총량 388 일치 = {total_regions_hit_sum == 388}")


① 팬덤 커버리지: 100/100 (지역 언급 0건인 팬덤: 0개)
② 검출 지역 수 총량(n_regions_hit 합계): 388 / 688 (목표 대비 56.4%)

문서 서술과 일치 여부:
  팬덤 커버리지 100/100 일치 = True
  검출 지역 수 총량 388 일치 = True



## 4. 지역별 전국 분포 — 어느 시/도가 가장 많이 언급되는가

100개 팬덤 전체를 합산해, 17개 시/도 중 실제로 K팝·트로트 팬덤 근거문장에
가장 많이 등장하는 지역이 어디인지 집계합니다(수도권 쏠림 여부를 눈으로
확인하기 위함 — 팬덤루트랩 상세명세서의 `regional_index` 스키마 설명에
언급된 "전국 합계 기준 서울 36%·부산 13%·대구 8%" 서술과 방향이 같은지
참고 비교합니다. 단, 그 서술은 최종 v7 리포트 기준이라 절대 수치는
다를 수 있습니다).


In [6]:

region_totals = {r: 0 for r in REGIONS}
for d in drp.values():
    for r in REGIONS:
        region_totals[r] += d["region_mention_counts"].get(r, 0)

grand_total = sum(region_totals.values())
region_dist_df = pd.DataFrame([
    {"지역": r, "총언급수": c, "비중": round(c / grand_total, 4)}
    for r, c in sorted(region_totals.items(), key=lambda x: -x[1])
])
print(f"전체 지역 언급 총계: {grand_total}건")
region_dist_df


전체 지역 언급 총계: 775건


,지역,총언급수,비중
0,서울,257,0.3316
1,부산,102,0.1316
2,대구,66,0.0852
3,인천,54,0.0697
4,경기,51,0.0658
5,광주,38,0.0490
6,경남,36,0.0465
7,경북,33,0.0426
8,강원,26,0.0335
9,전남,22,0.0284



## 5. 한계 (문서에서 이미 밝힌 것 재확인)

1. "언급 0건"은 "지역 연고가 실제로 없다"는 뜻이 아니라 "이 코퍼스에서
   검출되지 않았다"는 뜻입니다 — round22 최종 데이터도 100/100 커버리지에
   도달했지만, 이는 검출 지역 수(388)가 목표(688)에 크게 못 미치는
   상태에서의 100/100이라는 점에 유의해야 합니다(팬덤 커버리지와 지역
   총량 커버리지는 다른 지표입니다).
2. 부분 문자열 매칭 기반이라(정식 개체명 인식이 아님), 목록에 없는
   지역 별칭·표기는 누락될 수 있습니다.
3. 이 노트북은 문서 4절의 라운드별 갱신 이력 표(v7 8~22라운드)를
   재계산하지 않습니다 — 각 라운드 시점의 중간 스냅샷이 이번 세션에
   보존되어 있지 않고(라운드마다 domestic_regional_pilot_v6.json이
   덮어써지는 구조), 문서 4절의 표는 원본 서술을 그대로 표로 옮긴
   것입니다.
